# Rebuild the 2025 street-condition dataset

Run these cells from top to bottom in a fresh kernel. This workflow reads the saved monthly raw JSON files, checks them, and rebuilds the cleaned/labeled data. It does not call the live API or change the original CSVs. Outputs go to `data/rebuilt/`, so the existing EDA notebook can continue using its current input.

The helper functions are in `collection_pipeline.py`. The original interactive collection notebook is preserved in `archive/data_collection_original.ipynb` as a historical reference.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Support launching Jupyter from either the repository root or src/.
PROJECT_DIR = Path.cwd().resolve()
if (PROJECT_DIR / "collection_pipeline.py").is_file():
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / "src" / "collection_pipeline.py").is_file():
    raise FileNotFoundError("Launch Jupyter from the repository root or src/.")
sys.path.insert(0, str(PROJECT_DIR / "src"))

from collection_pipeline import build_dataset

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = DATA_DIR / "rebuilt"
print("Reading saved JSON from:", DATA_DIR)
print("Writing rebuilt CSVs to:", OUTPUT_DIR)


## 1. Process all twelve months

Each month is selected explicitly. The loader recognizes the existing February filename ending in `_2_raw.json`. It does not read monthly, combined, or labeled CSVs as collection inputs. Duplicate IDs or records outside the expected month/agency/problem cause an error. Missing closure dates are retained; negative closure durations are excluded.


In [ ]:
combined_data, labeled_data, monthly_checks = build_dataset(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    year=2025,
)

display(monthly_checks)
print("Combined shape:", combined_data.shape)
print("Labeled shape:", labeled_data.shape)


## 2. Verify the target

`closed_48h = 1` means a recorded closure from 0 through 48 hours inclusive. Later recorded closures receive 0. Missing-closure Pending/Open requests also receive 0 under the project's sufficient-follow-up and record-completeness assumptions. Unlabelable outcomes are excluded. This rule is for the saved 2025 cohort, not recent requests that have not had 48 hours of follow-up.

For the committed snapshot, expect 70,324 combined rows, 70,322 labeled rows, 38 labeled columns, 44,355 ones, and 25,967 zeros. All saved data are preserved; rebuilt rows are sorted by creation date and ID.


In [ ]:
print("Duplicate IDs:", labeled_data["unique_key"].duplicated().sum())
print("Missing targets:", labeled_data["closed_48h"].isna().sum())
print("Negative durations:", (labeled_data["closure_hours"] < 0).sum())
display(labeled_data["closed_48h"].value_counts())
display(labeled_data[["unique_key", "created_date", "closed_date", "closure_hours", "closed_48h"]].head())


## 3. Saved outputs and optional live collection

The rebuilt folder contains twelve cleaned monthly CSVs, `monthly_checks_2025.csv`, `street_conditions_2025_combined.csv`, and `street_conditions_2025_labeled.csv`. Rerunning replaces only those rebuilt outputs, not the original files in `data/`. The audit table now covers all twelve months.

`collect_month(start_date, end_date)` in the helper module remains available for a deliberate live API download. It uses `requests`, 5,000-record pagination, and the original DOT/Street Condition filters. It is not invoked by this notebook. Preserve any new download separately; the API changes over time, so a live download may not match the saved snapshot. Schema changes require review before rebuilding.

The data dictionary in `data/data_dictionary.md` explains all 38 columns and the limitations of the target. This rerun does not implement or change the EDA.
